In [1]:
#This script takes every simulation with all points and calculates rmse and mae in each case

In [6]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt

from matplotlib.pyplot import figure
import seaborn as sns
import matplotlib.gridspec as gridspec
import ast
import sys
sys.path.append('../no_degeneracy/')
sys.path.append('../no_degeneracy/Prior/')
from mcmc import *
from parallel import *
from fit_prior import read_prior_par
from sklearn.metrics import mean_squared_error
from sklearn.metrics import root_mean_squared_error
from sklearn.metrics import mean_absolute_error

In [7]:
def clean_index(dataframe):
    dataframe.set_index('Unnamed: 0', inplace=True)
    dataframe.index.name = None
    dataframe= dataframe.reset_index(drop=True)
    return dataframe
#-------------------------------------------------------------------------

#add predictions of bms to the dataframe. Acommodate for 2dimensional functions
def add_bms_pred(dataframe, bms_trace, number_param, dimensions=False):

    if dimensions==True:
         VARS = ['x','y',]
         prior_par = read_prior_par('../no_degeneracy/Prior/final_prior_param_sq.named_equations.nv2.np10.2016-09-09 18:49:42.600380.dat')
    else:
        VARS = ['x',]
        prior_par = read_prior_par('../no_degeneracy/Prior/final_prior_param_sq.named_equations.nv1.np10.2017-10-18 18:07:35.089658.dat')

    try:
        x = dataframe[[c for c in VARS]].copy()
        y=dataframe.z
    except KeyError:
        print("HOLA!")

        
        VARS = ['x1',]
        x = dataframe[[c for c in VARS]].copy()
        y=dataframe.y
    

    #mdl model
    minrow = bms_trace[bms_trace.H == min(bms_trace.H)].iloc[0]
    minH, minexpr, minparvals = minrow.H, minrow.expr, ast.literal_eval(minrow.parvals)

    
    t = Tree(
        variables=list(x.columns),
        parameters=['a%d' % i for i in range(number_param)],
        x=x, y=y,
        prior_par=prior_par,
        max_size=200,
        from_string=minexpr,
    )

    t.set_par_values(deepcopy(minparvals))

    dplot = deepcopy(dataframe)
    dplot['zbms'] = t.predict(x)


    return dplot
#-------------------------------------------------------------------------

#Append indices from the approximation dataset that need to be removed from the interpolation dataset
def rows_to_remove(function_name, index_length,step):
    n0=0 #initial index
    
    if function_name=='10':
        n_points_interpolation=int(np.sqrt(index_length))  #number of rows or columns in the interpolation grid
        n_points_row=int(n_points_interpolation/5)         #number of rows or columns in the approximation grid

        print(n_points_interpolation)
        print(n_points_row)
        
        for i in range(n_points_row):
            print(n0)
            a1=np.arange(n0,n0+n_points_interpolation,step)#Get indices in the interpolation column/row that exist in the approximation
            n0=a1[-1] + n_points_interpolation*4 + 5 #Hop over row/column indices of the interpolation grid that dont exist in the approximation grid
            
            try:
                approximation_indices=np.append(approximation_indices,a1) #append indices to remove from each set of columns/rows
            except NameError:
                approximation_indices=a1 #initialize array

    else:
        approximation_indices=np.arange(n0,n0+n_points_int,step)#Get indices in the interpolation column/row that exist in the approximation

    return approximation_indices

In [5]:
#Read NN and BMS data
functions=['1', '5' , '7', '8', '10']
realizations=2
sigmas=[0.0, 0.02, 0.04,0.06, 0.08, 0.1, 0.12, 0.14, 0.16, 0.18, 0.20]

runid=0
NPAR=10 #10, 20
steps=50000


interpolation_step=5
train_fraction=3/4

rmse_nn_train=[];rmse_nn_test=[]
rmse_mdl_train=[];rmse_mdl_test=[]
mae_nn_train=[];mae_nn_test=[]
mae_mdl_train=[];mae_mdl_test=[]

n_index=[];r_index=[];sigma_index=[];function_index=[]

#Put mae and rmse of each simulation (on nn and bms) in a dataframe
for function in functions:
    for sigma in sigmas:
        for r in range(realizations+1):
            
            #Save indices for dataframe
            r_index.append(r);sigma_index.append(sigma);function_index.append(function)
            
            #Read NN data: original and high resolution
            #------------------------------------------------------------------------------------------
            original_file_path='../../data/nns/nguyen/approximation/'
            model_d='NN_no_overfit_sigma_' + str(sigma) + '_r_' + str(r) + '.csv'
            d=pd.read_csv(original_file_path + model_d)

            hr_file_path='../../data/nns/nguyen/inter_extrapolation/'
            model_dhr='NN_no_overfit_inter_extrapolation_sigma_' + str(sigma) + '_r_' + str(r) + '.csv'
            dhr=pd.read_csv(hr_file_path + model_dhr)
            #------------------------------------------------------------------------------------------

            #High resolution dataset: run over functions
            #-------------------------------------------------------------
            dnhr=dhr[dhr['rep']==int(function)]
            dnhr=clean_index(dnhr)
            #-------------------------------------------------------------

            #Read BMS data
            #-------------------------------------------------------------
            trace_path='../../data/MSTraces/nguyen/'
            filename='BMS_nguyen_n_'+str(function)+'_sigma_'+str(sigma)+ '_r_' + str(r) + '_trace_'+str(steps)+'_prior_'+str(NPAR)+ '.csv'
            trace=pd.read_csv(trace_path + filename, sep=';', header=None, names=['t', 'H', 'expr', 'parvals', 'kk1', 'kk2','kk3'])
            #-------------------------------------------------------------
            
            #Add BMS prediction to high resolution dataframe
            #-------------------------------------------------------------
            #differentiate one and two-variable functions
            if function=='10':
                dinterpolate=add_bms_pred(dnhr, trace, NPAR, dimensions=True)
            else:
                dinterpolate=add_bms_pred(dnhr, trace, NPAR)
            #-------------------------------------------------------------
            
            #Remove points of approximation from the dataset
            #-------------------------------------------------------------
            dn=d[d['rep']==int(function)]
            dn=clean_index(dn)

            #n_points_app=int(len(dn.index))
            n_points_int=int(len(dinterpolate.index))
            approximation_indices=rows_to_remove(function,n_points_int,interpolation_step)

            dinterpolate=dinterpolate.drop(approximation_indices)
            display(dinterpolate)
            #-------------------------------------------------------------
            
            
            #Compute and save Errors to dataframe
            #-----------------------------------------------------------------------------------------------------------------------            
            n_points=int(len(dnhr.index))
            train_size_bms=int(n_points*train_fraction)           
            print(train_size_bms)

            
            #nn errors
            rmse_nn_train_i=root_mean_squared_error(dinterpolate.loc[:train_size_bms-1]['zmodel'],dinterpolate.loc[:train_size_bms -1]['z'])
            rmse_nn_train.append(rmse_nn_train_i)
                
            rmse_nn_test_i=root_mean_squared_error(dinterpolate.loc[train_size_bms-1:]['zmodel'],dinterpolate.loc[train_size_bms -1:]['z'])
            rmse_nn_test.append(rmse_nn_test_i)

            mae_nn_train_i=mean_absolute_error(dinterpolate.loc[:train_size_bms-1]['zmodel'],dinterpolate.loc[:train_size_bms -1]['z'])
            mae_nn_train.append(mae_nn_train_i)
            
            mae_nn_test_i=mean_absolute_error(dinterpolate.loc[train_size_bms-1:]['zmodel'],dinterpolate.loc[train_size_bms -1:]['z'])
            mae_nn_test.append(mae_nn_test_i)
        
            #bms errors
            try:
                rmse_mdl_train_i=root_mean_squared_error(dinterpolate.loc[:train_size_bms-1]['zbms'],dinterpolate.loc[:train_size_bms-1]['z'])
            except ValueError:
                rmse_mdl_train_i=np.inf
            rmse_mdl_train.append(rmse_mdl_train_i)

            try:
                rmse_mdl_test_i=root_mean_squared_error(dinterpolate.loc[train_size_bms-1:]['zbms'],dinterpolate.loc[train_size_bms-1:]['z'])
            except ValueError:
                rmse_mdl_test_i=np.inf
                    
            rmse_mdl_test.append(rmse_mdl_test_i)

            try:
                mae_mdl_train_i=mean_absolute_error(dinterpolate.loc[:train_size_bms-1]['zbms'],dinterpolate.loc[:train_size_bms-1]['z'])
            except ValueError:
                mae_mdl_train_i=np.inf
                    
            mae_mdl_train.append(mae_mdl_train_i)

            try:
                mae_mdl_test_i=mean_absolute_error(dinterpolate.loc[train_size_bms-1:]['zbms'],dinterpolate.loc[train_size_bms-1:]['z'])
            except ValueError:
                mae_mdl_test_i=np.inf
                
            mae_mdl_test.append(mae_mdl_test_i)
            #-----------------------------------------------------------------------------------------------------------------------
            

      
errors_df=pd.DataFrame({'sigma':sigma_index, 'function':function_index, 'mae_nn_interp.':mae_nn_train, 'mae_nn_extrap.':mae_nn_test, 
                        'mae_mdl_interp.':mae_mdl_train, 'mae_mdl_extrap.':mae_mdl_test, 'rmse_nn_interp.':rmse_nn_train, 
                        'rmse_nn_extrap.': rmse_nn_test, 'rmse_mdl_interp.':rmse_mdl_train, 'rmse_mdl_extrap.': rmse_mdl_test, 
                        'r': r_index})
errors_df.to_csv('../../data/all_errors_nguyen_interpolation' + '.csv')
display(errors_df)

,x,y,z,rep,zmodel,zbms
1,-0.995,0.0,-3.011651,1,-2.996476,-3.011651
2,-0.990,0.0,-2.973702,1,-2.961468,-2.973702
3,-0.985,0.0,-2.936150,1,-2.926631,-2.936150
4,-0.980,0.0,-2.898993,1,-2.891973,-2.898993
6,-0.970,0.0,-2.825853,1,-2.823222,-2.825853
...,...,...,...,...,...,...
394,0.970,0.0,6.815269,1,4.414315,6.815269
396,0.980,0.0,6.971089,1,4.450703,6.971089
397,0.985,0.0,7.049904,1,4.468549,7.049904
398,0.990,0.0,7.129326,1,4.486165,7.129326


300


/export/home/shared/Projects/ANN/Sergio/BMS_approximator/bin/nguyen_functions/../no_degeneracy/mcmc.py:654: OptimizeWarning: Covariance of the parameters could not be estimated
  res = curve_fit(


,x,y,z,rep,zmodel,zbms
1,-0.995,0.0,-3.011651,1,-2.998498,-3.011651
2,-0.990,0.0,-2.973702,1,-2.963414,-2.973702
3,-0.985,0.0,-2.936150,1,-2.928490,-2.936150
4,-0.980,0.0,-2.898993,1,-2.893735,-2.898993
6,-0.970,0.0,-2.825853,1,-2.824762,-2.825853
...,...,...,...,...,...,...
394,0.970,0.0,6.815269,1,4.106371,6.815269
396,0.980,0.0,6.971089,1,4.136431,6.971089
397,0.985,0.0,7.049904,1,4.151177,7.049904
398,0.990,0.0,7.129326,1,4.165736,7.129326


300


...

,x,y,z,rep,zmodel,zbms
1,0.0000,0.0025,0.000000,10,-0.003326,0.000000
2,0.0000,0.0050,0.000000,10,-0.003359,0.000000
3,0.0000,0.0075,0.000000,10,-0.003392,0.000000
4,0.0000,0.0100,0.000000,10,-0.003424,0.000000
6,0.0000,0.0150,0.000000,10,-0.003489,0.000000
...,...,...,...,...,...,...
159995,0.9975,0.9875,0.878116,10,0.988273,0.968467
159996,0.9975,0.9900,0.877525,10,0.987788,0.968337
159997,0.9975,0.9925,0.876932,10,0.987303,0.968206
159998,0.9975,0.9950,0.876338,10,0.986816,0.968076


120000
400
80
0
2000
4000
6000
8000
10000
12000
14000
16000
18000
20000
22000
24000
26000
28000
30000
32000
34000
36000
38000
40000
42000
44000
46000
48000
50000
52000
54000
56000
58000
60000
62000
64000
66000
68000
70000
72000
74000
76000
78000
80000
82000
84000
86000
88000
90000
92000
94000
96000
98000
100000
102000
104000
106000
108000
110000
112000
114000
116000
118000
120000
122000
124000
126000
128000
130000
132000
134000
136000
138000
140000
142000
144000
146000
148000
150000
152000
154000
156000
158000


,x,y,z,rep,zmodel,zbms
1,0.0000,0.0025,0.000000,10,-0.025479,0.000000
2,0.0000,0.0050,0.000000,10,-0.025157,0.000000
3,0.0000,0.0075,0.000000,10,-0.024839,0.000000
4,0.0000,0.0100,0.000000,10,-0.024525,0.000000
6,0.0000,0.0150,0.000000,10,-0.023910,0.000000
...,...,...,...,...,...,...
159995,0.9975,0.9875,0.878116,10,0.984484,0.952571
159996,0.9975,0.9900,0.877525,10,0.984299,0.952266
159997,0.9975,0.9925,0.876932,10,0.984115,0.951959
159998,0.9975,0.9950,0.876338,10,0.983931,0.951651


120000


,sigma,function,mae_nn_interp.,mae_nn_extrap.,mae_mdl_interp.,mae_mdl_extrap.,rmse_nn_interp.,rmse_nn_extrap.,rmse_mdl_interp.,rmse_mdl_extrap.,r
0,0.00,1,0.003155,0.852872,1.233750e-16,6.661338e-16,0.004471,1.167822,1.968682e-16,8.514391e-16,0
1,0.00,1,0.003693,0.998788,1.183226e-16,4.797260e-16,0.004808,1.342675,2.084592e-16,7.405598e-16,1
2,0.00,1,0.002434,0.865053,1.411848e-16,8.662481e-16,0.003358,1.172886,2.380712e-16,1.114327e-15,2
3,0.02,1,0.010924,0.783363,3.273081e-03,2.350146e-01,0.013970,1.052788,6.553932e-03,2.940264e-01,0
4,0.02,1,0.004217,0.704566,1.770215e-03,1.752805e-02,0.005219,0.984614,2.444490e-03,1.886998e-02,1
...,...,...,...,...,...,...,...,...,...,...,...
160,0.18,10,0.004011,0.008843,6.389056e-03,2.007255e-02,0.005232,0.011983,9.177947e-03,3.037298e-02,1
161,0.18,10,0.008255,0.046167,5.506491e-03,1.378513e-02,0.010680,0.058829,7.237618e-03,2.012844e-02,2
162,0.20,10,0.007934,0.025464,5.719325e-03,2.097733e-02,0.010095,0.030987,6.870082e-03,2.536005e-02,0
163,0.20,10,0.003555,0.052856,4.559083e-03,1.973315e-02,0.004999,0.058217,6.785228e-03,2.852211e-02,1
